# 05 — Investment Scoring

Calculates LAD-level investment scores for the Airbnb Investment Intelligence App.

Reads from:
- airbnb_app.gold.lad_summary

Writes to:
- airbnb_app.gold.investment_scores_lad

Purpose:
Create a transparent scoring model that ranks areas based on Airbnb revenue, occupancy, long-term rental comparison, house prices, and market activity.

##Load the data

In [0]:
from pyspark.sql.functions import col, min, max, lit, round

lad = spark.table("airbnb_app.gold.lad_summary")
display(lad)

## 1. Select Scoring Metrics

The score uses LAD-level summary metrics from the gold layer.

Current factors:
- Airbnb revenue potential
- Occupancy proxy
- Short-term rental yield
- Yield gap between Airbnb and long-term rent
- Market saturation

In [0]:
score_base = lad.select(
    "lad_code",
    "lad_name",
    "_city",
    "listing_count",
    "median_airbnb_income_calendar",
    "median_airbnb_income_ons",
    "median_annual_rent",
    "median_house_price",
    "median_occupancy_proxy",
    "str_gross_yield",
    "ltr_gross_yield",
    "yield_gap"
)

## 2. Normalise Metrics

Each metric is converted to a 0–100 scale using min-max normalisation.

For market saturation, the score is reversed because more listings means more competition.

In [0]:
stats = score_base.select(
    min("median_airbnb_income_calendar").alias("min_income"),
    max("median_airbnb_income_calendar").alias("max_income"),
    min("median_occupancy_proxy").alias("min_occ"),
    max("median_occupancy_proxy").alias("max_occ"),
    min("str_gross_yield").alias("min_str_yield"),
    max("str_gross_yield").alias("max_str_yield"),
    min("yield_gap").alias("min_gap"),
    max("yield_gap").alias("max_gap"),
    min("listing_count").alias("min_listings"),
    max("listing_count").alias("max_listings")
).collect()[0]

In [0]:
scored = score_base.withColumn(
    "revenue_score",
    ((col("median_airbnb_income_calendar") - lit(stats["min_income"])) /
     (lit(stats["max_income"]) - lit(stats["min_income"]))) * 100
).withColumn(
    "occupancy_score",
    ((col("median_occupancy_proxy") - lit(stats["min_occ"])) /
     (lit(stats["max_occ"]) - lit(stats["min_occ"]))) * 100
).withColumn(
    "str_yield_score",
    ((col("str_gross_yield") - lit(stats["min_str_yield"])) /
     (lit(stats["max_str_yield"]) - lit(stats["min_str_yield"]))) * 100
).withColumn(
    "yield_gap_score",
    ((col("yield_gap") - lit(stats["min_gap"])) /
     (lit(stats["max_gap"]) - lit(stats["min_gap"]))) * 100
).withColumn(
    "saturation_score",
    100 - (
        ((col("listing_count") - lit(stats["min_listings"])) /
         (lit(stats["max_listings"]) - lit(stats["min_listings"]))) * 100
    )
)

## 3. Calculate Investment Score

The Investment Score is calculated using weighted scoring:

- Revenue Score = 30%
- Occupancy Score = 25%
- STR Yield Score = 25%
- Yield Gap Score = 10%
- Market Saturation Score = 10%

The score is rounded to one decimal place.

In [0]:
investment_scores = scored.withColumn(
    "investment_score",
    round(
        (col("revenue_score") * 0.30) +
        (col("occupancy_score") * 0.25) +
        (col("str_yield_score") * 0.25) +
        (col("yield_gap_score") * 0.10) +
        (col("saturation_score") * 0.10),
        1
    )
)

## 4. Display Ranked Areas

Areas are ranked from highest to lowest Investment Score.

In [0]:
display(
    investment_scores.orderBy(col("investment_score").desc())
)

## Aggregate to Neighbourhood Level

Listings are grouped by neighbourhood so the app can compare areas rather than individual properties.

In [0]:
area_scores = df.groupBy("neighbourhood_cleansed").agg(
    avg("estimated_revenue").alias("avg_revenue"),
    avg("occupancy_proxy").alias("avg_occupancy"),
    avg("review_scores_rating").alias("avg_rating")
)

## Normalise Scores

Each metric is converted to a 0–100 scale using min-max normalisation.

This allows revenue, occupancy and rating to be combined fairly.

In [0]:
stats = area_scores.select(
    min("avg_revenue").alias("min_rev"),
    max("avg_revenue").alias("max_rev"),
    min("avg_occupancy").alias("min_occ"),
    max("avg_occupancy").alias("max_occ"),
    min("avg_rating").alias("min_rating"),
    max("avg_rating").alias("max_rating")
).collect()[0]

area_scores = area_scores.withColumn(
    "revenue_score",
    ((col("avg_revenue") - lit(stats["min_rev"])) /
     (lit(stats["max_rev"]) - lit(stats["min_rev"]))) * 100
).withColumn(
    "occupancy_score",
    ((col("avg_occupancy") - lit(stats["min_occ"])) /
     (lit(stats["max_occ"]) - lit(stats["min_occ"]))) * 100
).withColumn(
    "rating_score",
    ((col("avg_rating") - lit(stats["min_rating"])) /
     (lit(stats["max_rating"]) - lit(stats["min_rating"]))) * 100
)

## Calculate Investment Score

The final Investment Score is calculated using weighted scoring:

- Revenue Score = 40%
- Occupancy Score = 40%
- Rating Score = 20%

The final score is rounded to one decimal place.

In [0]:
area_scores = area_scores.withColumn(
    "investment_score",
    round(
        (col("revenue_score") * 0.4) +
        (col("occupancy_score") * 0.4) +
        (col("rating_score") * 0.2),
        1
    )
)

## Display Results

Neighbourhoods are ranked from highest to lowest investment score.

In [0]:
display(area_scores.orderBy(col("investment_score").desc()))

In [0]:
display(
    investment_scores.select(
        "str_gross_yield",
        "ltr_gross_yield",
        "yield_gap"
    )
)

In [0]:
display(
    investment_scores.orderBy(col("investment_score").desc())
)